# Portolan Colab MCP server

Google アカウントだけで完結する Portolan コレクションの作成・公開・AI から読める化を検証するノートブック。

- 認証: Google アカウント (Colab の OAuth)
- Drive: マウントで Google アカウントのストレージへ書き込み
- GitHub Pages: MCP ツール経由で repository へ commit (push で GitHub Actions が自動デプロイ)

GitHub の fine-grained token を **前もって** 1 つ作っておいてください (`Contents: Read and write`、対象 repository を選択)。

以下のセルを上から順に実行してください。

## 1. セットアップと認証 (Google アカウント)

In [ ]:
!pip install -q fastmcp geopandas
from google.colab import drive
drive.mount('/content/drive')
print('Googleアカウントで認証されました')

## 2. GitHub トークンの設定

実行時にパスワード入力欄でトークンを貼り付けてください。入力内容はセル表示には残りません。

In [ ]:
import getpass, os
os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub fine-grained token: ')
print('token set:', bool(os.environ['GITHUB_TOKEN']))

## 3. server.py を取得して MCP ツールを読み込む

リポジトリ内の `server.py` を使います。実行環境を汚さないよう tools だけ import します。

In [ ]:
!git clone --depth 1 https://github.com/watanabe3tipapa/portolan-sandbox.git
%cd /content/portolan-sandbox/demos/colab-mcp-server

## 4. 機能確認: 公開済みデモコレクションを読む

In [ ]:
from server import read_collection
print(read_collection('https://watanabe3tipapa.github.io/portolan-sandbox/demo-collection/collection.json'))

## 5. Drive に書き込み (Google アカウントのストレージ)

生成した `collection.json` や `AGENTS.md` を Drive へ退避します。

In [ ]:
from server import write_to_drive
print(write_to_drive('Portolan/demo-collection/collection.json', open('fixtures/collection.json').read()))

## 6. GitHub Pages にデプロイ (MCP ツール)

ローカルの git clone なしで、Contents API 経由でリポジトリのファイルを更新します。
更新を push すると GitHub Actions が自動で GitHub Pages へ反映します。

In [ ]:
from server import deploy_to_github_pages

msg = "Update demo collection from Colab (MCP tool)"
print(deploy_to_github_pages(
    repo='watanabe3tipapa/portolan-sandbox',
    file_path='portolan-lp/public/demo-collection/collection.json',
    content=open('fixtures/collection.json').read(),
    message=msg,
))

## 7. 結果の確認

約 1 分後に以下の URL で README / collection.json が読めるはずです。

In [ ]:
import time, urllib.request, json

def check(url):
    for _ in range(12):
        try:
            with urllib.request.urlopen(url) as r:
                print(r.status, url)
                return
        except Exception:
            time.sleep(5)
    print('timeout', url)

check('https://watanabe3tipapa.github.io/portolan-sandbox/demo-collection/collection.json')
check('https://watanabe3tipapa.github.io/portolan-sandbox/demo-collection/README.md')